# EC Number Classification

Train a logistic regression on top of the reaction embeddings to predict EC enzyme class (top-level 1–7). Evaluate with stratified 5-fold cross-validation and compare against a random-embedding baseline.

> Requires `uv sync --extra all`, plus:
> - `data/embeddings/medium/{embeddings.npy,smarts.txt}` — download from [Zenodo](https://doi.org/10.5281/zenodo.22645328), or regenerate (see README's Data pipeline and Training sections).
> - `data/raw/retrorules-v3.0-{metanetx,rhea}.csv` — download from [retrorules.org](https://retrorules.org/) (for EC-class labels).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

## Load embeddings and labels

In [ ]:
EMB_PATH    = "data/embeddings/medium/embeddings.npy"
SMARTS_PATH = "data/embeddings/medium/smarts.txt"
RAW_FILES   = [
    "data/raw/retrorules-v3.0-metanetx.csv",
    "data/raw/retrorules-v3.0-rhea.csv",
]

embeddings  = np.load(EMB_PATH).astype(np.float32)
smarts_list = Path(SMARTS_PATH).read_text().splitlines()

raw = pd.concat([pd.read_csv(f) for f in RAW_FILES], ignore_index=True)
raw = raw[["TEMPLATE", "ECS"]].dropna(subset=["TEMPLATE", "ECS"]).drop_duplicates("TEMPLATE")
raw["ec1"] = raw["ECS"].str.split(";").str[0].str.split(".").str[0]

df = pd.DataFrame({"smarts": smarts_list, "emb_idx": range(len(smarts_list))})
df = df.merge(raw[["TEMPLATE", "ec1"]].rename(columns={"TEMPLATE": "smarts"}),
              on="smarts", how="inner")
df = df[df["ec1"].isin([str(i) for i in range(1, 8)])]

X = embeddings[df["emb_idx"].values]
y = df["ec1"].values

print(f"Labelled reactions: {len(X):,}")
print(pd.Series(y).value_counts().sort_index().rename("count").to_string())

## Subsample for speed

5-fold CV on 361k reactions is slow. Subsample to 30k (stratified) for a quick benchmark; remove the cap for full results.

In [ ]:
N_SAMPLES = 30_000   # set to None to use the full dataset

if N_SAMPLES and len(X) > N_SAMPLES:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X), size=N_SAMPLES, replace=False)
    X_sub, y_sub = X[idx], y[idx]
else:
    X_sub, y_sub = X, y

le = LabelEncoder().fit(y_sub)
y_enc = le.transform(y_sub)

print(f"Using {len(X_sub):,} reactions  —  {len(le.classes_)} classes")

## 5-fold cross-validation

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs",
                                   multi_class="multinomial", n_jobs=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipeline, X_sub, y_enc, cv=cv,
                        scoring=["accuracy", "f1_macro"],
                        return_train_score=True)

print(f"Val  accuracy : {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
print(f"Val  F1 macro : {scores['test_f1_macro'].mean():.3f} ± {scores['test_f1_macro'].std():.3f}")
print(f"Train accuracy: {scores['train_accuracy'].mean():.3f} ± {scores['train_accuracy'].std():.3f}")

## Random-embedding baseline

Replaces the pretrained embeddings with random Gaussian vectors of the same shape. Any accuracy above this comes from the model's learned representations.

In [ ]:
X_random = np.random.default_rng(0).standard_normal(X_sub.shape).astype(np.float32)

scores_random = cross_validate(pipeline, X_random, y_enc, cv=cv,
                               scoring=["accuracy", "f1_macro"])

print(f"Random baseline accuracy : {scores_random['test_accuracy'].mean():.3f} ± {scores_random['test_accuracy'].std():.3f}")
print(f"Random baseline F1 macro : {scores_random['test_f1_macro'].mean():.3f} ± {scores_random['test_f1_macro'].std():.3f}")

## Per-class report and confusion matrix

In [ ]:
EC_NAMES = {
    "1": "Oxidoreductases", "2": "Transferases", "3": "Hydrolases",
    "4": "Lyases",          "5": "Isomerases",  "6": "Ligases",
    "7": "Translocases",
}

from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_sub, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

pipeline.fit(X_tr, y_tr)
y_pred = pipeline.predict(X_te)

target_names = [f"EC {c} — {EC_NAMES[c]}" for c in le.classes_]
print(classification_report(y_te, y_pred, target_names=target_names))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred,
    display_labels=[f"EC {c}" for c in le.classes_],
    normalize="true",
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("Normalised confusion matrix — EC top-level class")
plt.tight_layout()
plt.show()

## CV accuracy by fold

In [ ]:
folds = np.arange(1, 6)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(folds - 0.2, scores["test_accuracy"],        0.35, label="pretrained", color="steelblue")
ax.bar(folds + 0.2, scores_random["test_accuracy"], 0.35, label="random",    color="lightcoral")
ax.set_xticks(folds)
ax.set_xticklabels([f"Fold {i}" for i in folds])
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("EC classification accuracy per fold")
plt.tight_layout()
plt.show()